# 07 — Merge Colab Chroma ke Local
Notebook ini dipakai setiap kali selesai embed dokumen baru di Google Colab.
Fungsinya merge hasil Chroma dari Colab ke vector_db/ local tanpa hapus data lama.

**Kapan dipakai:**
- Selesai embed dokumen baru di Colab

## 0. Config
Isi path zip file hasil download dari Google Colab.

In [4]:
import os

# ← GANTI INI setiap kali download zip baru dari Colab
ZIP_PATH = r"C:\Users\Dev\Downloads\chroma_manufacturing.zip"
# "C:\Users\Dev\Downloads\chroma_db.zip"
# Path extract sementara
EXTRACT_PATH = r"C:\Users\Dev\Downloads\chroma_temp"

# Path vector_db local (jangan diubah)
LOCAL_VECTOR_DB = "../vector_db"

print(f"ZIP source  : {ZIP_PATH}")
print(f"Extract ke  : {EXTRACT_PATH}")
print(f"Local DB    : {LOCAL_VECTOR_DB}")
print(f"\nZip exists  : {os.path.exists(ZIP_PATH)}")

ZIP source  : C:\Users\Dev\Downloads\chroma_manufacturing.zip
Extract ke  : C:\Users\Dev\Downloads\chroma_temp
Local DB    : ../vector_db

Zip exists  : True


## 1. Extract ZIP
Extract file zip dari Colab ke folder sementara.

In [5]:
import zipfile
import shutil

# Hapus folder temp kalau sudah ada
if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)

# Extract zip
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print(f"Extracted ke : {EXTRACT_PATH}")
print(f"Isi folder   :")
for f in os.listdir(EXTRACT_PATH):
    print(f"  - {f}")

Extracted ke : C:\Users\Dev\Downloads\chroma_temp
Isi folder   :
  - chroma_db


## 2. Merge ke Local Vector DB
Ambil semua vectors dari Colab dan tambahkan ke local Chroma.
Data lama tidak akan dihapus!

In [6]:
import chromadb

# Load Chroma dari Colab (hasil extract)
colab_path = os.path.join(EXTRACT_PATH, "chroma_db")
colab_client = chromadb.PersistentClient(path=colab_path)
colab_col = colab_client.get_collection("phis_sds")

print(f"Vectors dari Colab : {colab_col.count()}")

# Load Chroma local
local_client = chromadb.PersistentClient(path=LOCAL_VECTOR_DB)
local_col = local_client.get_or_create_collection(
    name="phis_sds",
    metadata={"hnsw:space": "cosine"}
)

print(f"Vectors local saat ini : {local_col.count()}")

# Ambil semua data dari Colab
data = colab_col.get(include=["embeddings", "documents", "metadatas"])

# Add ke local
local_col.add(
    ids=data["ids"],
    embeddings=data["embeddings"],
    documents=data["documents"],
    metadatas=data["metadatas"]
)

print(f"\nSelesai merge!")
print(f"Total vectors sekarang : {local_col.count()}")
print(f"(SAM: 666 + Manufacturing: {local_col.count() - 666})")

Vectors dari Colab : 459
Vectors local saat ini : 666

Selesai merge!
Total vectors sekarang : 1125
(SAM: 666 + Manufacturing: 459)


## 3. Verifikasi
Test search dari kedua dokumen sekaligus.

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="thenlper/gte-large")
vectorstore = Chroma(
    collection_name="phis_sds",
    embedding_function=embeddings,
    persist_directory=LOCAL_VECTOR_DB
)

print(f"Total vectors : {vectorstore._collection.count()}")
print()

# Test query SAM
print("=== Query SAM ===")
docs = vectorstore.similarity_search("SAM approval process", k=2)
for doc in docs:
    print(f"Page {doc.metadata.get('page')} | Source: {doc.metadata.get('source', 'SAM')}")
    print(f"{doc.page_content[:100]}")
    print()

# Test query Manufacturing
print("=== Query Manufacturing ===")
docs = vectorstore.similarity_search("manufacturing process pharmacy", k=2)
for doc in docs:
    print(f"Page {doc.metadata.get('page')} | Source: {doc.metadata.get('source', 'N/A')}")
    print(f"{doc.page_content[:100]}")
    print()

C:\Users\Dev\AppData\Local\Temp\ipykernel_21392\3996176361.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="thenlper/gte-large")
C:\Users\Dev\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4677.52it/s]
C:\Users\Dev\AppData\Local\Temp\ipykernel_21392\3996176361.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.

Total vectors : 1125

=== Query SAM ===
Page 107 | Source: ../data/PHIS_SDS_SAM_1.1.pdf
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 96
Post-Condition System shall display SAM 

Page 66 | Source: ../data/PHIS_SDS_SAM_1.1.pdf
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 55
vii. Approval by Hospital Director/ Dist

=== Query Manufacturing ===
Page 13 | Source: PhIS_SDS_Manufacturing_GP_1.0.2.pdf
3.2. System Transaction Design 
3.2.1. Galenical & Prepacking Manufacturing 
3.2.1.1. Work Order 
 


Page 10 | Source: PhIS_SDS_Manufacturing_GP_1.0.2.pdf
the system, guiding the development team in translating requirements into practical, 
implementable 

